# Lab 3 – Exploratory Data Analysis (EDA)
**Dataset:** Medical Insurance Cost (`insurance.csv`)  
**Goal:** Understand and visualise the data before modelling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')

## 1. Load Dataset

In [ ]:
df = pd.read_csv('insurance.csv')
print(f'Shape: {df.shape}')
df.head()

## 2. Descriptive Statistics

In [ ]:
df.describe(include='all').round(2)

## 3. Missing Values Check

In [ ]:
missing = df.isnull().sum()
pct     = (missing / len(df) * 100).round(2)
result  = pd.DataFrame({'Count': missing, 'Percentage (%)': pct})
print(result if missing.sum() > 0 else 'No missing values — dataset is complete.')

## 4. Categorical Feature Value Counts

In [ ]:
for col in ['sex', 'smoker', 'region']:
    print(f'--- {col} ---')
    print(df[col].value_counts())
    print()

## 5. Distribution of Numerical Features

In [ ]:
num_cols = ['age', 'bmi', 'children', 'charges']
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=30, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(f'Distribution of {col}', fontsize=11)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')

plt.suptitle('Numerical Feature Distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 6. Charges by Smoker Status

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.histplot(data=df, x='charges', hue='smoker', bins=40,
             palette={'yes':'tomato','no':'steelblue'}, ax=axes[0], alpha=0.7)
axes[0].set_title('Charges Distribution: Smoker vs Non-Smoker')

sns.boxplot(data=df, x='smoker', y='charges',
            palette={'yes':'tomato','no':'steelblue'}, ax=axes[1])
axes[1].set_title('Charges Box Plot by Smoker Status')

plt.tight_layout()
plt.show()

print(df.groupby('smoker')['charges'].agg(['mean','median','std']).round(2))

## 7. Charges by Region and Sex

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.boxplot(data=df, x='region', y='charges', palette='Set2', ax=axes[0])
axes[0].set_title('Charges by Region')
axes[0].tick_params(axis='x', rotation=15)

sns.boxplot(data=df, x='sex', y='charges', palette='Set1', ax=axes[1])
axes[1].set_title('Charges by Sex')

plt.tight_layout()
plt.show()

## 8. Scatter: BMI vs Charges (coloured by Smoker)

In [ ]:
plt.figure(figsize=(9, 6))
colors = df['smoker'].map({'yes':'tomato','no':'steelblue'})
plt.scatter(df['bmi'], df['charges'], c=colors, alpha=0.5, edgecolors='none', s=30)
plt.xlabel('BMI')
plt.ylabel('Charges (USD)')
plt.title('BMI vs Insurance Charges')
from matplotlib.lines import Line2D
legend_elements = [Line2D([0],[0],marker='o',color='w',markerfacecolor='tomato',markersize=9,label='Smoker'),
                   Line2D([0],[0],marker='o',color='w',markerfacecolor='steelblue',markersize=9,label='Non-Smoker')]
plt.legend(handles=legend_elements)
plt.tight_layout()
plt.show()

## 9. Scatter: Age vs Charges (coloured by Smoker)

In [ ]:
plt.figure(figsize=(9, 6))
plt.scatter(df['age'], df['charges'], c=colors, alpha=0.5, edgecolors='none', s=30)
plt.xlabel('Age')
plt.ylabel('Charges (USD)')
plt.title('Age vs Insurance Charges')
from matplotlib.lines import Line2D
legend_elements = [Line2D([0],[0],marker='o',color='w',markerfacecolor='tomato',markersize=9,label='Smoker'),
                   Line2D([0],[0],marker='o',color='w',markerfacecolor='steelblue',markersize=9,label='Non-Smoker')]
plt.legend(handles=legend_elements)
plt.tight_layout()
plt.show()

## 10. Correlation Heatmap

In [ ]:
df_encoded = df.copy()
df_encoded['smoker_enc'] = (df['smoker'] == 'yes').astype(int)
df_encoded['sex_enc']    = (df['sex']    == 'male').astype(int)

corr_cols = ['age','sex_enc','bmi','children','smoker_enc','charges']
corr = df_encoded[corr_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            linewidths=0.5, cbar_kws={'shrink':0.8})
plt.title('Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.show()

## 11. Pair Plot

In [ ]:
sns.pairplot(df[['age','bmi','charges','smoker']], hue='smoker',
             palette={'yes':'tomato','no':'steelblue'}, diag_kind='kde', height=2.5)
plt.suptitle('Pair Plot — Insurance Dataset', y=1.02, fontsize=13)
plt.show()

## 12. Key EDA Findings

- **Smoker status** is by far the strongest driver of charges. Smokers pay on average **3–4× more** than non-smokers.
- **BMI × Smoker interaction**: smokers with BMI ≥ 30 form a distinct high-cost cluster.
- **Age** has a moderate positive correlation with charges (older → more expensive).
- **Region** and **sex** show minimal impact on charges individually.
- **Charges are right-skewed** — log transformation or robust models may help.
- **No missing values** detected; dataset is clean and ready for preprocessing.